# Precision functional mapping

Infomap algorithm
(Gordon 2017)

Networks were defined in each individual by collapsing across density thresholds (Laumann et al., 2015) and assigning identities based on similarity to a set of template networks (Figure S3A).

- cross-correlation matrix of the time courses from all brain vertices (voxels for sucortical regions)
- Correlations between vertices/voxels within 30 mm of each other were set to zero in this matrix to avoid basing network membership on correlations attributable to spatial smoothing (Geodesic distance was used for within-hemisphere surface connections)
- thresholded at a range of values calculated based on the resulting density of the matrix; the density thresholds ranged from 0.3% to 5%
- thresholded matrices were used as inputs for the Infomap algorithm, which calculated community assignments (representing brain networks) separately for each threshold. Small networks with 400 or fewer vertices / voxels were considered unassigned and removed from further consideration

In [1]:
import nilearn
import numpy as np
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as op
from brainspace.utils.parcellation import map_to_labels

bids_folder = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'

? parcel of vertex space ?


In [47]:
sub ='01'
source_folder = op.join(bids_folder, 'derivatives', 'correlation_matrices.tryNoHalo') # .parcel
confspec = '32Pscrub3BPfilter'
cm_file = op.join(source_folder,f'sub-{sub}_ses-1_task-magjudge_confspec-{confspec}runFD104-6runs_CM-unfiltered.npy')
cm_f = np.load(cm_file)

cm_cc_mask = op.join(bids_folder, 'derivatives',f'gradients.{confspec}runFD104', f'sub-{sub}', f'sub-{sub}_cc-mask_space-fsaverag5.npy')
cm_cc_mask = np.load(cm_cc_mask)
cm_filtered = cm_f[cm_cc_mask, :][:, cm_cc_mask]


In [48]:
# Threshold matrix
# recheck code!

conn_matrix =cm_filtered

# Optional: Threshold or binarize
# For example, keep top 10% strongest connections
def threshold_matrix(mat, proportion=0.3):
    n = mat.shape[0]
    mat = mat.copy()
    # Remove diagonal
    np.fill_diagonal(mat, 0)
    # Find threshold value
    thresh = np.percentile(mat[mat > 0], 100 - 100 * proportion)
    mat[mat < thresh] = 0
    return mat

conn_thresh = threshold_matrix(conn_matrix)
conn_thresh

array([[0.        , 0.        , 0.30436216, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.30436216, 0.        , 0.        , ..., 0.        , 0.        ,
        0.18523411],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.58771879,
        0.1705411 ],
       [0.        , 0.        , 0.        , ..., 0.58771879, 0.        ,
        0.58323743],
       [0.        , 0.        , 0.18523411, ..., 0.1705411 , 0.58323743,
        0.        ]])

In [49]:
from infomap import Infomap

mat = conn_thresh

N = mat.shape[0]
im = Infomap()  # add flags like '--two-level' if needed

for i in range(N):
    for j in range(i+1, N):
        w = mat[i, j]
        if w > 0:
            im.add_link(i, j, w)

im.run()
#modules = {node.node_id: node.module_id for node in im.nodes}
modules = [[node.node_id, node.module_id] for node in im.nodes]
print(np.unique(np.array(modules)[:,1]))

  Infomap v2.8.0 starts at 2025-07-07 10:14:01
  -> Input network: 
  -> No file output!
  OpenMP 201511 detected with 16 threads...
  -> Ordinary network input, using the Map Equation for first order network flows
Calculating global network flow using flow model 'undirected'... 
  -> Using undirected links.
  => Sum node flow: 1, sum link flow: 1
Build internal network with 18711 nodes and 36205736 links...
  -> One-level codelength: 13.9562336

Trial 1/1 starting at 2025-07-07 10:14:32
Two-level compression: 0.33% 0.061% 
Partitioned to codelength 0.165745651 + 13.7360609 = 13.90180651 in 5 modules.
Super-level compression: to codelength 13.90180651 in 5 top modules.

Recursive sub-structure compression: 0% . Found 2 levels with codelength 13.90180651

=> Trial 1/1 finished in 137.502492s with codelength 13.9018065


Summary after 1 trial
Best end modular solution in 2 levels:
Per level number of modules:         [          5,           0] (sum: 5)
Per level number of leaf nodes:    

In [50]:
print("Node → Module mapping:", modules)

Node → Module mapping: [[414, 1], [6546, 1], [4900, 1], [8917, 1], [8916, 1], [2318, 1], [7862, 1], [7865, 1], [2198, 1], [17424, 1], [9286, 1], [5205, 1], [15254, 1], [15253, 1], [16192, 1], [3556, 1], [93, 1], [15392, 1], [2216, 1], [11051, 1], [415, 1], [7067, 1], [13084, 1], [13468, 1], [13699, 1], [9779, 1], [10141, 1], [10003, 1], [16767, 1], [4602, 1], [13716, 1], [7492, 1], [13565, 1], [16845, 1], [5203, 1], [1701, 1], [3320, 1], [16837, 1], [6924, 1], [2872, 1], [2801, 1], [18500, 1], [8119, 1], [15332, 1], [13082, 1], [13030, 1], [7488, 1], [1369, 1], [16524, 1], [8974, 1], [1047, 1], [7999, 1], [18137, 1], [15403, 1], [11916, 1], [14968, 1], [15255, 1], [10787, 1], [5871, 1], [4364, 1], [7323, 1], [7489, 1], [9790, 1], [11265, 1], [5126, 1], [16840, 1], [16839, 1], [6926, 1], [6705, 1], [15016, 1], [637, 1], [13080, 1], [16844, 1], [10156, 1], [11477, 1], [6623, 1], [17432, 1], [7490, 1], [8817, 1], [14072, 1], [2803, 1], [18498, 1], [8307, 1], [16201, 1], [16843, 1], [6549,

In [52]:
from numrisk.fmri_analysis.gradients.utils import get_basic_mask

mask, labeling_noParcel = get_basic_mask()
mask[mask == True] = cm_cc_mask 

[get_dataset_dir] Dataset found in /home/ubuntu/nilearn_data/destrieux_surface


In [61]:
module_mapping = np.array(modules)

# Use -1 as fill value instead of np.nan since module IDs are integers
modules_fsav5 = map_to_labels(module_mapping[:,1], labeling_noParcel, mask=mask, fill=-1)

In [ ]:
surf_map = modules_fsav5

from  nilearn.datasets import fetch_surf_fsaverage
import nilearn.plotting as nplt 
fsaverage = fetch_surf_fsaverage('fsaverage5') 
views = ['medial','lateral','dorsal','posterior']
cmap = 'viridis'


for i, hemi in enumerate(['L','R']):
    map = np.split(surf_map,2)[i]
    surf_mesh = fsaverage.infl_right if hemi =='R' else fsaverage.infl_left
    bg_map = fsaverage.sulc_right if hemi =='R' else fsaverage.sulc_left

    figure, axes = plt.subplots(nrows=1, ncols=len(views),figsize = (15,8), subplot_kw=dict(projection='3d'))
    for i,view in enumerate(views):
        colbar = True if view == 'posterior' else False
        nplt.plot_surf(surf_mesh=surf_mesh , surf_map= map, # infl_right # pial_right
                view= view,cmap=cmap, colorbar=colbar, #title=f'sub-{sub}, grad {n_grad+1}',
                bg_map=bg_map, bg_on_data=True,darkness=0.7, axes=axes[i]) 
    figure.subplots_adjust(wspace=0.01)
    #figure.suptitle(f'{cm_name}, grad {n_grad+1}', y=0.75)